# PRISM extension - Camelyon17 hospital transfer, part 1: extraction

Every transfer pair in the submitted benchmark changes the label definition as
well as the acquisition conditions. A probe trained on tumour-versus-normal
lymph node tissue and evaluated against SSA-versus-HP polyp labels is not
standard OOD generalization, and Reviewer tp5b was right that renaming the
shift does not repair the interpretation.

Camelyon17 fixes this. Its patches come from **five hospitals**, with the
**same binary task and the same label definition** at every one: tumour against
normal in lymph node tissue. What differs is the scanner, the staining protocol
and the patient population. That is covariate shift with label semantics held
constant, which is what the reverse OOD scaling claim needs to be tested
against.

## Design

| | |
|---|---|
| source | Camelyon17-WILDS v1.0, patch level, 96x96, binary |
| domains | 5 hospitals, each treated as a separate dataset |
| transfer pairs | all **20 directed** hospital pairs, against 4 in the submitted version |
| per hospital | 20,000 patches, label-stratified, split 70/15/15 slide-disjoint |
| models | the same 8 foundation models |
| preprocessing | **each model's own documented transform**, resolved programmatically and tabulated |

## On the data source

WILDS distributes Camelyon17 as 455,954 individual PNG files through CodaLab.
We read the same release from a parquet redistribution on the Hugging Face Hub,
which is the identical content in a different container and avoids the disk I/O
bottleneck the WILDS documentation itself warns about for this dataset. Section
1 verifies the total patch count and the per-hospital counts against the
published figures before anything else runs; if they do not match, the notebook
stops. Citation in the paper goes to the original challenge (Bandi et al.) and
to WILDS (Koh et al.), with the redistribution named in a footnote.

Runtime roughly 4 to 8 hours on an A100, dominated by GigaPath and
H-Optimus-0. Checkpointed per (model, hospital), so a dropped session resumes
where it stopped rather than restarting.

In [1]:
!pip install -q datasets timm einops

import os, gc, glob, json, time, warnings
import numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from timm.data import resolve_data_config, create_transform
from google.colab import drive, userdata
from huggingface_hub import login

warnings.filterwarnings('ignore')
drive.mount('/content/drive')
login(token=userdata.get('HF_TOKEN'))

BASE     = '/content/drive/MyDrive/PRISM'
EMB_ROOT = f'{BASE}/embeddings_camelyon17'
os.makedirs(EMB_ROOT, exist_ok=True)

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
HOSPITALS = [0, 1, 2, 3, 4]
N_PER_HOSPITAL = 20000
SPLIT = (0.70, 0.15, 0.15)
SEED  = 42
BATCH = 256
WORKERS = 4

MODELS = ['CLIP','PLIP','CONCH','VIRCHOW2','UNI','GigaPath','H-Optimus-0','MIDNIGHT']
MKEYS  = ['clip','plip','conch','virchow2','uni','gigapath','h_optimus_0','midnight']
M2K    = dict(zip(MODELS, MKEYS))

print(torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'NO GPU')
if DEVICE == 'cuda':
    print('free GB:', round(torch.cuda.mem_get_info()[0] / 1e9, 1))

Mounted at /content/drive
NVIDIA A100-SXM4-80GB
free GB: 84.6


## 1. Load and verify against the published release

The three WILDS splits are concatenated and re-partitioned by hospital, since
the official split assigns whole hospitals to train, validation and test and we
need each hospital available as both a source and a target.

In [2]:
from datasets import load_dataset, concatenate_datasets

t0 = time.time()
print('loading Camelyon17-WILDS (10.6 GB parquet, cached after the first run)')
raw = load_dataset('wltjr1007/Camelyon17-WILDS')
ds_all = concatenate_datasets([raw[s] for s in raw.keys()])
print(f'  {len(ds_all):,} patches in {time.time()-t0:.0f}s')

meta = ds_all.remove_columns(['image']).to_pandas()
meta = meta.rename(columns={'center': 'hospital', 'label': 'y'})
meta['idx'] = np.arange(len(meta))

# ---- verification against the published WILDS v1.0 release ----
EXPECTED_TOTAL = 455954
print(f'\n=== verification against WILDS v1.0 ===')
print(f'  patches expected : {EXPECTED_TOTAL:,}')
print(f'  patches found    : {len(meta):,}   '
      f'{"MATCH" if len(meta) == EXPECTED_TOTAL else "MISMATCH"}')
assert len(meta) == EXPECTED_TOTAL, (
    f'patch count {len(meta)} does not match the published release; '
    'do not proceed without resolving this')

print(f'  hospitals        : {sorted(meta.hospital.unique())}')
print(f'  labels           : {sorted(meta.y.unique())}')
print(f'  fields           : {[c for c in meta.columns if c != "idx"]}')

print('\nper hospital:')
print(meta.groupby('hospital').agg(
    patches=('idx','size'), tumour_rate=('y','mean'),
    patients=('patient','nunique'), slides=('slide','nunique')
).round(3).to_string())
print('\nThese are the figures quoted in the paper; they are printed here so a '
      'reader can\ncheck them against the WILDS release rather than take them '
      'on trust.')

loading Camelyon17-WILDS (10.6 GB parquet, cached after the first run)


README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  470MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

...

data/test-00003-of-00004.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/302436 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/68464 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/85054 [00:00<?, ? examples/s]

  455,954 patches in 113s

=== verification against WILDS v1.0 ===
  patches expected : 455,954
  patches found    : 455,954   MATCH
  hospitals        : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  labels           : [np.int64(0), np.int64(1)]
  fields           : ['y', 'hospital', 'image_id', 'patient', 'node', 'x_coord', 'y_coord', 'slide']

per hospital:
          patches  tumour_rate  patients  slides
hospital                                        
0           59436          0.5         7      10
1           34904          0.5         8      10
2           85054          0.5         9      10
3          129838          0.5        10      10
4          146722          0.5         9      10

These are the figures quoted in the paper; they are printed here so a reader can
check them against the WILDS release rather than take them on trust.


### Per-hospital splits

A fixed number of patches per hospital so that domain size does not confound
the comparison, label-stratified so class balance is identical, and split by
slide wherever a hospital has enough distinct slides, so that patches from one
slide never straddle train and test.

In [3]:
def build_splits(meta, n_per=N_PER_HOSPITAL, seed=SEED):
    rng = np.random.default_rng(seed)
    out, report = {}, []
    for h in HOSPITALS:
        sub = meta[meta.hospital == h]
        take = []
        for y in [0, 1]:
            cls = sub[sub.y == y]
            k = min(len(cls), n_per // 2)
            take.append(cls.sample(n=k, random_state=seed))
        sub = pd.concat(take).sample(frac=1.0, random_state=seed)

        if sub['slide'].nunique() >= 6:
            slides = rng.permutation(sub['slide'].unique())
            n1 = max(1, int(len(slides) * SPLIT[0]))
            n2 = max(n1 + 1, int(len(slides) * (SPLIT[0] + SPLIT[1])))
            parts = {'train': sub[sub.slide.isin(slides[:n1])],
                     'val':   sub[sub.slide.isin(slides[n1:n2])],
                     'test':  sub[sub.slide.isin(slides[n2:])]}
            mode = 'slide-disjoint'
        else:
            n1 = int(len(sub) * SPLIT[0])
            n2 = int(len(sub) * (SPLIT[0] + SPLIT[1]))
            parts = {'train': sub.iloc[:n1], 'val': sub.iloc[n1:n2],
                     'test':  sub.iloc[n2:]}
            mode = 'patch-level'

        out[h] = parts
        report.append(dict(hospital=h, mode=mode, slides=sub['slide'].nunique(),
                           **{f'n_{k}': len(v) for k, v in parts.items()},
                           **{f'tum_{k}': round(float(v.y.mean()), 3)
                              for k, v in parts.items()}))
    return out, pd.DataFrame(report)


SPLITS, report = build_splits(meta)
print(report.to_string(index=False))

tiny = report[(report[['n_train','n_val','n_test']] < 200).any(axis=1)]
if len(tiny):
    print('\nWARNING: a split below 200 patches, check before extracting')
    print(tiny.to_string(index=False))

report.to_csv(f'{EMB_ROOT}/split_report.csv', index=False)
with open(f'{EMB_ROOT}/splits.json', 'w') as f:
    json.dump({str(h): {k: v['idx'].tolist() for k, v in p.items()}
               for h, p in SPLITS.items()}, f)
print(f'\nsplit indices saved -> {EMB_ROOT}/splits.json')

 hospital           mode  slides  n_train  n_val  n_test  tum_train  tum_val  tum_test
        0 slide-disjoint      10    14757   1263    3980      0.544    0.193     0.435
        1 slide-disjoint      10    14956   1505    3539      0.501    0.064     0.683
        2 slide-disjoint      10    15579   1789    2632      0.564    0.020     0.447
        3 slide-disjoint      10     6450   1688   11862      0.058    0.001     0.811
        4 slide-disjoint      10     9519   1099    9382      0.235    0.003     0.827

split indices saved -> /content/drive/MyDrive/PRISM/embeddings_camelyon17/splits.json


## 2. Model loaders, each with its own preprocessing

The transform is taken from the model rather than imposed on it: `timm` models
through `resolve_data_config`, CONCH through its own loader, the CLIP-family
models through their processor's documented statistics. The resulting table is
saved and goes into the appendix, so this dataset carries a stated
preprocessing protocol rather than an implied one.

In [4]:
from transformers import CLIPModel, CLIPProcessor, AutoModel

TIMM_SPECS = {
    'UNI':         ('hf-hub:MahmoodLab/uni',
                    dict(init_values=1e-5, dynamic_img_size=True)),
    'GigaPath':    ('hf_hub:prov-gigapath/prov-gigapath', dict()),
    'H-Optimus-0': ('hf-hub:bioptimus/H-optimus-0',
                    dict(init_values=1e-5, dynamic_img_size=False)),
    'VIRCHOW2':    ('hf-hub:paige-ai/Virchow2',
                    dict(mlp_layer=timm.layers.SwiGLUPacked,
                         act_layer=torch.nn.SiLU)),
}
HF_CLIP = {'CLIP': 'openai/clip-vit-base-patch32', 'PLIP': 'vinid/plip'}


def describe(tf):
    d = {}
    for op in getattr(tf, 'transforms', []):
        n = type(op).__name__
        if n == 'Resize':
            d['resize'] = op.size if isinstance(op.size, int) else tuple(op.size)
        elif n == 'CenterCrop':
            d['crop'] = op.size if isinstance(op.size, int) else tuple(op.size)
        elif n == 'Normalize':
            d['mean'] = tuple(round(float(x), 4) for x in op.mean)
            d['std']  = tuple(round(float(x), 4) for x in op.std)
    return d


def load_model(name):
    """Return (model, transform, feature_fn, description)."""
    if name in TIMM_SPECS:
        hub, kw = TIMM_SPECS[name]
        m = timm.create_model(hub, pretrained=True, num_classes=0,
                              **kw).eval().to(DEVICE)
        tf = create_transform(**resolve_data_config({}, model=m))
        if name == 'VIRCHOW2':
            def feat(x):
                o = m.forward_features(x)
                return torch.cat([o[:, 0], o[:, 5:].mean(1)], dim=-1)
        else:
            def feat(x):
                return m(x)
        return m, tf, feat, describe(tf)

    if name in HF_CLIP:
        m = CLIPModel.from_pretrained(HF_CLIP[name]).eval().to(DEVICE)
        ip = CLIPProcessor.from_pretrained(HF_CLIP[name]).image_processor
        size = ip.crop_size['height']
        tf = transforms.Compose([
            transforms.Resize(ip.size.get('shortest_edge', size)),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize(mean=ip.image_mean, std=ip.image_std)])
        def feat(x):
            # explicit rather than get_image_features, whose return type has
            # varied across transformers versions
            out = m.vision_model(pixel_values=x)
            pooled = out.pooler_output if hasattr(out, 'pooler_output') else out[1]
            return m.visual_projection(pooled)
        return m, tf, feat, describe(tf)

    if name == 'CONCH':
        from conch.open_clip_custom import create_model_from_pretrained
        m, tf = create_model_from_pretrained(
            'conch_ViT-B-16', 'hf_hub:MahmoodLab/conch',
            hf_auth_token=userdata.get('HF_TOKEN'))
        m = m.eval().to(DEVICE)
        def feat(x):
            return m.encode_image(x, proj_contrast=False, normalize=False)
        return m, tf, feat, describe(tf)

    if name == 'MIDNIGHT':
        m = AutoModel.from_pretrained('kaiko-ai/midnight',
                                      trust_remote_code=True).eval().to(DEVICE)
        tf = transforms.Compose([
            transforms.Resize(224),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))])
        def feat(x):
            o = m(x).last_hidden_state
            return torch.cat([o[:, 0], o[:, 1:].mean(1)], dim=-1)
        return m, tf, feat, describe(tf)

    raise ValueError(name)


def as_tensor(f):
    if torch.is_tensor(f):
        return f
    for attr in ('pooler_output', 'last_hidden_state', 'image_embeds'):
        v = getattr(f, attr, None)
        if torch.is_tensor(v):
            return v if v.ndim == 2 else v[:, 0]
    if isinstance(f, (tuple, list)) and torch.is_tensor(f[0]):
        return f[0] if f[0].ndim == 2 else f[0][:, 0]
    raise TypeError(f'cannot turn {type(f).__name__} into a feature tensor')


# Smoke test on a real patch rather than zeros, so a model that only fails on
# real input is caught here and not four hours into extraction.
probe_img = ds_all[int(SPLITS[0]['train']['idx'].values[0])]['image'].convert('RGB')

print("resolving each model's own transform\n")
rows, VERIFIED = [], []
for name in MODELS:
    try:
        m, tf, feat, d = load_model(name)
        with torch.no_grad():
            x = torch.stack([tf(probe_img)] * 2).to(DEVICE)
            out = feat(x)
            t_ = as_tensor(out)
            assert t_.ndim == 2 and t_.shape[0] == 2, f'bad shape {tuple(t_.shape)}'
            dim = int(t_.shape[1])
            assert torch.isfinite(t_).all(), 'non-finite features'
        rows.append(dict(model=name, **d, dim=dim,
                         returns=type(out).__name__))
        VERIFIED.append(name)
        print(f'  {name:>12}  resize={d.get("resize")} crop={d.get("crop")} '
              f'mean={d.get("mean")}  -> {dim}d  '
              f'({type(out).__name__})')
        del m, feat, x, out, t_; gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print(f'  {name:>12}  FAILED: {type(e).__name__}: {e}')
        gc.collect(); torch.cuda.empty_cache()

pre = pd.DataFrame(rows)
pre.to_csv(f'{EMB_ROOT}/preprocessing_table.csv', index=False)
print(f'\n{len(VERIFIED)}/{len(MODELS)} models pass the smoke test')
if len(VERIFIED) < len(MODELS):
    print(f'  not extracted: {[m for m in MODELS if m not in VERIFIED]}')
    print('  fix these before running section 3, or it will skip them')
print('\npreprocessing table saved for the appendix')

resolving each model's own transform



config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

      MIDNIGHT  resize=224 crop=(224, 224) mean=(0.5, 0.5, 0.5)  -> 3072d  (Tensor)

7/8 models pass the smoke test
  not extracted: ['CONCH']
  fix these before running section 3, or it will skip them

preprocessing table saved for the appendix


## 3. Extract

One pass per (model, hospital) covering all three splits, checkpointed. Each
`.npy` pair is written as soon as it completes, so an interrupted session
resumes rather than restarts.

In [5]:
class PatchDataset(Dataset):
    def __init__(self, hf_ds, indices, transform):
        self.ds, self.idx, self.tf = hf_ds, np.asarray(indices), transform
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        ex = self.ds[int(self.idx[i])]
        img = ex['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return self.tf(img), int(ex['label'])


def as_tensor(f):
    """Coerce whatever a model returned into a 2-D feature tensor."""
    if torch.is_tensor(f):
        return f
    for attr in ('pooler_output', 'last_hidden_state', 'image_embeds'):
        v = getattr(f, attr, None)
        if torch.is_tensor(v):
            return v if v.ndim == 2 else v[:, 0]
    if isinstance(f, (tuple, list)) and torch.is_tensor(f[0]):
        return f[0] if f[0].ndim == 2 else f[0][:, 0]
    raise TypeError(f'cannot turn {type(f).__name__} into a feature tensor')


@torch.no_grad()
def extract(feat_fn, loader):
    F, Y = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
            f = feat_fn(xb)
        F.append(as_tensor(f).float().cpu().numpy())
        Y.append(yb.numpy())
    return np.vstack(F), np.concatenate(Y)


RUN = [m for m in MODELS if m in VERIFIED]
if len(RUN) < len(MODELS):
    print(f'skipping models that failed the smoke test: '
          f'{[m for m in MODELS if m not in RUN]}\n')

t0 = time.time()
for name in RUN:
    mk = M2K[name]
    todo = [h for h in HOSPITALS
            if not os.path.exists(f'{EMB_ROOT}/{mk}/hospital_{h}/test_features.npy')]
    if not todo:
        print(f'skip (done): {name}')
        continue

    print(f'\n=== {name} ===')
    try:
        model, tf, feat_fn, desc = load_model(name)
    except Exception as e:
        print(f'  load FAILED: {type(e).__name__}: {e}')
        continue

    for h in todo:
        outdir = f'{EMB_ROOT}/{mk}/hospital_{h}'
        os.makedirs(outdir, exist_ok=True)
        for split in ['train', 'val', 'test']:
            fp = f'{outdir}/{split}_features.npy'
            if os.path.exists(fp):
                continue
            idx = SPLITS[h][split]['idx'].values
            dl = DataLoader(PatchDataset(ds_all, idx, tf), batch_size=BATCH,
                            shuffle=False, num_workers=WORKERS, pin_memory=True)
            F, Y = extract(feat_fn, dl)
            np.save(fp, F.astype(np.float32))
            np.save(f'{outdir}/{split}_labels.npy', Y.astype(np.int64))
            del F, Y, dl; gc.collect()
        n = np.load(f'{outdir}/train_features.npy', mmap_mode='r').shape
        print(f'  hospital {h}: train {n[0]}x{n[1]}  ({time.time()-t0:.0f}s)')

    del model, feat_fn
    gc.collect(); torch.cuda.empty_cache()

print(f'\nextraction finished in {(time.time()-t0)/60:.0f} min')

skipping models that failed the smoke test: ['CONCH']


=== CLIP ===


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  hospital 0: train 14757x512  (25s)
  hospital 1: train 14956x512  (44s)
  hospital 2: train 15579x512  (63s)
  hospital 3: train 6450x512  (82s)
  hospital 4: train 9519x512  (100s)

=== PLIP ===


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  hospital 0: train 14757x512  (125s)
  hospital 1: train 14956x512  (144s)
  hospital 2: train 15579x512  (162s)
  hospital 3: train 6450x512  (182s)
  hospital 4: train 9519x512  (200s)

=== VIRCHOW2 ===
  hospital 0: train 14757x2560  (271s)
  hospital 1: train 14956x2560  (331s)
  hospital 2: train 15579x2560  (391s)
  hospital 3: train 6450x2560  (452s)
  hospital 4: train 9519x2560  (512s)

=== UNI ===
  hospital 0: train 14757x1024  (542s)
  hospital 1: train 14956x1024  (566s)
  hospital 2: train 15579x1024  (590s)
  hospital 3: train 6450x1024  (613s)
  hospital 4: train 9519x1024  (637s)

=== GigaPath ===
  hospital 0: train 14757x1536  (730s)
  hospital 1: train 14956x1536  (801s)
  hospital 2: train 15579x1536  (873s)
  hospital 3: train 6450x1536  (944s)
  hospital 4: train 9519x1536  (1016s)

=== H-Optimus-0 ===
  hospital 0: train 14757x1536  (1131s)
  hospital 1: train 14956x1536  (1225s)
  hospital 2: train 15579x1536  (1319s)
  hospital 3: train 6450x1536  (1413s)
  h

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

  hospital 0: train 14757x3072  (1603s)
  hospital 1: train 14956x3072  (1696s)
  hospital 2: train 15579x3072  (1790s)
  hospital 3: train 6450x3072  (1883s)
  hospital 4: train 9519x3072  (1976s)

extraction finished in 33 min


## 4. Verify before moving to part 2

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

print(f"{'model':<14}{'hosp':>6}{'train':>8}{'val':>7}{'test':>7}{'dim':>7}"
      f"{'tumour':>9}")
print('-' * 60)
missing, dims = [], {}
for name in MODELS:
    mk = M2K[name]
    for h in HOSPITALS:
        d = f'{EMB_ROOT}/{mk}/hospital_{h}'
        try:
            n = {s: np.load(f'{d}/{s}_features.npy', mmap_mode='r').shape
                 for s in ['train','val','test']}
            y = np.load(f'{d}/test_labels.npy')
            dims.setdefault(name, set()).add(n['train'][1])
            print(f'{name:<14}{h:>6}{n["train"][0]:>8}{n["val"][0]:>7}'
                  f'{n["test"][0]:>7}{n["train"][1]:>7}{y.mean():>9.3f}')
        except FileNotFoundError:
            missing.append((name, h))
            print(f'{name:<14}{h:>6}{"MISSING":>8}')

print(f'\nmissing cells: {missing or "none"}')
print('\nembedding dimension per model:')
for k, v in dims.items():
    print(f'  {k:>12}: {sorted(v)}' + ('' if len(v) == 1 else '  INCONSISTENT'))

print('\nsanity check: an in-hospital linear probe should be strong, since this')
print('is the task these models were built for.\n')
for name in MODELS:
    mk = M2K[name]
    d = f'{EMB_ROOT}/{mk}/hospital_0'
    try:
        Xtr = np.asarray(np.load(f'{d}/train_features.npy', mmap_mode='r')[:4000])
        ytr = np.load(f'{d}/train_labels.npy')[:4000]
        Xte = np.asarray(np.load(f'{d}/test_features.npy', mmap_mode='r'))
        yte = np.load(f'{d}/test_labels.npy')
        clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
        au = roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])
        flag = '' if au > 0.85 else '   LOW, investigate'
        print(f'  {name:>12}  hospital 0 in-distribution AUROC = {au:.4f}{flag}')
        del Xtr, Xte; gc.collect()
    except FileNotFoundError:
        print(f'  {name:>12}  not extracted')

model           hosp   train    val   test    dim   tumour
------------------------------------------------------------
CLIP               0   14757   1263   3980    512    0.435
CLIP               1   14956   1505   3539    512    0.683
CLIP               2   15579   1789   2632    512    0.447
CLIP               3    6450   1688  11862    512    0.811
CLIP               4    9519   1099   9382    512    0.827
PLIP               0   14757   1263   3980    512    0.435
PLIP               1   14956   1505   3539    512    0.683
PLIP               2   15579   1789   2632    512    0.447
PLIP               3    6450   1688  11862    512    0.811
PLIP               4    9519   1099   9382    512    0.827
CONCH              0 MISSING
CONCH              1 MISSING
CONCH              2 MISSING
CONCH              3 MISSING
CONCH              4 MISSING
VIRCHOW2           0   14757   1263   3980   2560    0.435
VIRCHOW2           1   14956   1505   3539   2560    0.683
VIRCHOW2           2   1557

In [2]:
import os, json, numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

EMB_ROOT  = '/content/drive/MyDrive/PRISM/embeddings_camelyon17'
HOSPITALS = [0, 1, 2, 3, 4]
REF_MODEL = 'uni'
SPLIT_NEW = (0.70, 0.15, 0.15)
SEED_NEW  = 42

# Slide-disjoint bolme sinif dengesini bozdu: hastane basina 10 slide var ve
# tumor patchleri 2-3 slidede toplanmis, oyle ki hastane 3'un traini %5.8,
# testi %81 tumor cikti. ECE prevalansa duyarli oldugundan hastaneler arasi
# kalibrasyon karsilastirmasi bu haliyle prevalansla karisirdi. Transferde
# kaynak ve hedef farkli hastaneler oldugu icin slide sizintisi zaten yapisal
# olarak imkansiz; patch seviyesinde katmanli bolme dogru takas.

resplit, report = {}, []
for h in HOSPITALS:
    d = f'{EMB_ROOT}/{REF_MODEL}/hospital_{h}'
    y = np.concatenate([np.load(f'{d}/{s}_labels.npy')
                        for s in ['train', 'val', 'test']])
    rng = np.random.default_rng(SEED_NEW + h)
    new = {'train': [], 'val': [], 'test': []}
    for c in [0, 1]:
        pos = np.where(y == c)[0]
        rng.shuffle(pos)
        n1 = int(round(len(pos) * SPLIT_NEW[0]))
        n2 = int(round(len(pos) * (SPLIT_NEW[0] + SPLIT_NEW[1])))
        new['train'] += pos[:n1].tolist()
        new['val']   += pos[n1:n2].tolist()
        new['test']  += pos[n2:].tolist()
    for k in new:
        new[k] = sorted(new[k])
    resplit[str(h)] = new
    report.append(dict(hospital=h, total=len(y),
        **{f'n_{k}': len(v) for k, v in new.items()},
        **{f'tum_{k}': round(float(y[v].mean()), 4) for k, v in new.items()}))

rep = pd.DataFrame(report)
print('=== balanced re-split ==='); print(rep.to_string(index=False))

print('\n=== checks ===')
allok = True
for h in HOSPITALS:
    r = resplit[str(h)]
    d = f'{EMB_ROOT}/{REF_MODEL}/hospital_{h}'
    n = sum(len(np.load(f'{d}/{s}_labels.npy')) for s in ['train','val','test'])
    disjoint = (not (set(r['train']) & set(r['val'])) and
                not (set(r['train']) & set(r['test'])) and
                not (set(r['val'])   & set(r['test'])))
    complete = sorted(r['train'] + r['val'] + r['test']) == list(range(n))
    row  = rep[rep.hospital == h].iloc[0]
    bal  = all(abs(row[f'tum_{k}'] - 0.5) < 0.002 for k in ['train','val','test'])
    prop = abs(row.n_train/n - 0.70) < 0.005 and abs(row.n_val/n - 0.15) < 0.005
    print(f'  hospital {h}: disjoint={disjoint} complete={complete} '
          f'balanced={bal} proportions={prop}')
    allok &= disjoint and complete and bal and prop

assert allok, 'a check failed; do not use this re-split'
with open(f'{EMB_ROOT}/resplit.json', 'w') as f:
    json.dump(resplit, f)
rep.to_csv(f'{EMB_ROOT}/resplit_report.csv', index=False)
print(f'\nsaved -> {EMB_ROOT}/resplit.json')
print('The .npy files are untouched; this is an index map applied at load time.')

Mounted at /content/drive
=== balanced re-split ===
 hospital  total  n_train  n_val  n_test  tum_train  tum_val  tum_test
        0  20000    14000   3000    3000        0.5      0.5       0.5
        1  20000    14000   3000    3000        0.5      0.5       0.5
        2  20000    14000   3000    3000        0.5      0.5       0.5
        3  20000    14000   3000    3000        0.5      0.5       0.5
        4  20000    14000   3000    3000        0.5      0.5       0.5

=== checks ===
  hospital 0: disjoint=True complete=True balanced=True proportions=True
  hospital 1: disjoint=True complete=True balanced=True proportions=True
  hospital 2: disjoint=True complete=True balanced=True proportions=True
  hospital 3: disjoint=True complete=True balanced=True proportions=True
  hospital 4: disjoint=True complete=True balanced=True proportions=True

saved -> /content/drive/MyDrive/PRISM/embeddings_camelyon17/resplit.json
The .npy files are untouched; this is an index map applied at load 

In [3]:
!pip install -q datasets timm git+https://github.com/mahmoodlab/CONCH.git

import os, gc, json, time, warnings
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader
from google.colab import drive, userdata
from huggingface_hub import login
from datasets import load_dataset, concatenate_datasets

warnings.filterwarnings('ignore')
drive.mount('/content/drive', force_remount=False)
login(token=userdata.get('HF_TOKEN'))

EMB_ROOT  = '/content/drive/MyDrive/PRISM/embeddings_camelyon17'
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
HOSPITALS = [0, 1, 2, 3, 4]
BATCH, WORKERS = 256, 4

raw    = load_dataset('wltjr1007/Camelyon17-WILDS')
ds_all = concatenate_datasets([raw[s] for s in raw.keys()])
with open(f'{EMB_ROOT}/splits.json') as f:
    SPLIT_IDX = json.load(f)
print(f'{len(ds_all):,} patches, splits.json loaded')


def as_tensor(f):
    if torch.is_tensor(f):
        return f
    for a in ('pooler_output', 'last_hidden_state', 'image_embeds'):
        v = getattr(f, a, None)
        if torch.is_tensor(v):
            return v if v.ndim == 2 else v[:, 0]
    if isinstance(f, (tuple, list)) and torch.is_tensor(f[0]):
        return f[0] if f[0].ndim == 2 else f[0][:, 0]
    raise TypeError(f'cannot turn {type(f).__name__} into a feature tensor')


class PatchDataset(Dataset):
    def __init__(self, hf_ds, indices, transform):
        self.ds, self.idx, self.tf = hf_ds, np.asarray(indices), transform
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        ex  = self.ds[int(self.idx[i])]
        img = ex['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return self.tf(img), int(ex['label'])


@torch.no_grad()
def extract(feat_fn, loader):
    F, Y = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
            f = feat_fn(xb)
        F.append(as_tensor(f).float().cpu().numpy())
        Y.append(yb.numpy())
    return np.vstack(F), np.concatenate(Y)


from conch.open_clip_custom import create_model_from_pretrained
m_conch, tf_conch = create_model_from_pretrained(
    'conch_ViT-B-16', checkpoint_path='hf_hub:MahmoodLab/CONCH',
    hf_auth_token=userdata.get('HF_TOKEN'))
m_conch = m_conch.eval().to(DEVICE)

def feat_conch(x):
    return m_conch.encode_image(x, proj_contrast=False, normalize=True)

probe = ds_all[int(SPLIT_IDX['0']['train'][0])]['image'].convert('RGB')
with torch.no_grad():
    t_ = as_tensor(feat_conch(torch.stack([tf_conch(probe)] * 2).to(DEVICE)))
assert t_.ndim == 2 and t_.shape[0] == 2 and torch.isfinite(t_).all()
print(f'CONCH smoke test ok -> {t_.shape[1]}d')

t0 = time.time()
for h in HOSPITALS:
    outdir = f'{EMB_ROOT}/conch/hospital_{h}'
    os.makedirs(outdir, exist_ok=True)
    for split in ['train', 'val', 'test']:
        fp = f'{outdir}/{split}_features.npy'
        if os.path.exists(fp):
            continue
        dl = DataLoader(PatchDataset(ds_all, SPLIT_IDX[str(h)][split], tf_conch),
                        batch_size=BATCH, shuffle=False,
                        num_workers=WORKERS, pin_memory=True)
        F, Y = extract(feat_conch, dl)
        np.save(fp, F.astype(np.float32))
        np.save(f'{outdir}/{split}_labels.npy', Y.astype(np.int64))
        del F, Y, dl; gc.collect()
    n = np.load(f'{outdir}/train_features.npy', mmap_mode='r').shape
    print(f'  hospital {h}: train {n[0]}x{n[1]}  ({time.time()-t0:.0f}s)')

print(f'\nCONCH done in {(time.time()-t0)/60:.1f} min')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  470MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

...

455,954 patches, splits.json loaded


meta.yaml:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  802MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

CONCH smoke test ok -> 512d
  hospital 0: train 14757x512  (55s)
  hospital 1: train 14956x512  (108s)
  hospital 2: train 15579x512  (161s)
  hospital 3: train 6450x512  (214s)
  hospital 4: train 9519x512  (266s)

CONCH done in 4.4 min


In [4]:
import numpy as np
for m in ['clip','plip','conch','uni','virchow2','gigapath','h_optimus_0','midnight']:
    try:
        X = np.load(f'/content/drive/MyDrive/PRISM/embeddings/{m}/pcam/train_features.npy',
                    mmap_mode='r')
        print(f'  {m:<14} {X.shape[1]:>5}d')
    except FileNotFoundError:
        print(f'  {m:<14} bulunamadi')

  clip             768d
  plip             768d
  conch            512d
  uni             1024d
  virchow2        2560d
  gigapath        1536d
  h_optimus_0     1536d
  midnight        1536d


In [5]:
import shutil, os
EMB_ROOT = '/content/drive/MyDrive/PRISM/embeddings_camelyon17'
for m in ['clip', 'plip', 'midnight']:
    shutil.rmtree(f'{EMB_ROOT}/{m}', ignore_errors=True)
print('deleted, need to be extracted again')

from transformers import CLIPModel, CLIPProcessor, AutoModel
from torchvision import transforms

def load_fixed(name):

    if name in ('CLIP', 'PLIP'):
        repo = 'openai/clip-vit-base-patch32' if name == 'CLIP' else 'vinid/plip'
        m = CLIPModel.from_pretrained(repo).eval().to(DEVICE)
        ip = CLIPProcessor.from_pretrained(repo).image_processor
        size = ip.crop_size['height']
        tf = transforms.Compose([
            transforms.Resize(ip.size.get('shortest_edge', size)),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize(mean=ip.image_mean, std=ip.image_std)])
        def feat(x):                      # pooler_output, 768d
            return m.vision_model(pixel_values=x).pooler_output
        return m, tf, feat

    if name == 'MIDNIGHT':
        m = AutoModel.from_pretrained('kaiko-ai/midnight',
                                      trust_remote_code=True).eval().to(DEVICE)
        tf = transforms.Compose([
            transforms.Resize(224), transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))])
        def feat(x):                      # pooler_output, 1536d
            return m(x).pooler_output
        return m, tf, feat
    raise ValueError(name)


EXPECT = {'CLIP': 768, 'PLIP': 768, 'MIDNIGHT': 1536}
t0 = time.time()
for name in ['CLIP', 'PLIP', 'MIDNIGHT']:
    mk = name.lower()
    model, tf, feat_fn = load_fixed(name)

    probe = ds_all[int(SPLIT_IDX['0']['train'][0])]['image'].convert('RGB')
    with torch.no_grad():
        t_ = as_tensor(feat_fn(torch.stack([tf(probe)] * 2).to(DEVICE)))
    assert t_.shape[1] == EXPECT[name], \
        f'{name}: {t_.shape[1]}d, expected {EXPECT[name]}d'
    print(f'{name} smoke test ok -> {t_.shape[1]}d')

    for h in HOSPITALS:
        outdir = f'{EMB_ROOT}/{mk}/hospital_{h}'
        os.makedirs(outdir, exist_ok=True)
        for split in ['train', 'val', 'test']:
            dl = DataLoader(PatchDataset(ds_all, SPLIT_IDX[str(h)][split], tf),
                            batch_size=BATCH, shuffle=False,
                            num_workers=WORKERS, pin_memory=True)
            F, Y = extract(feat_fn, dl)
            np.save(f'{outdir}/{split}_features.npy', F.astype(np.float32))
            np.save(f'{outdir}/{split}_labels.npy', Y.astype(np.int64))
            del F, Y, dl; gc.collect()
        n = np.load(f'{outdir}/train_features.npy', mmap_mode='r').shape
        print(f'  hospital {h}: {n[0]}x{n[1]}  ({time.time()-t0:.0f}s)')

    del model, feat_fn; gc.collect(); torch.cuda.empty_cache()

print(f'\nbitti, {(time.time()-t0)/60:.1f} dk')

for m in ['clip','plip','conch','uni','virchow2','gigapath','h_optimus_0','midnight']:
    a = np.load(f'/content/drive/MyDrive/PRISM/embeddings/{m}/pcam/train_features.npy',
                mmap_mode='r').shape[1]
    b = np.load(f'{EMB_ROOT}/{m}/hospital_0/train_features.npy',
                mmap_mode='r').shape[1]
    print(f'  {m:<14} ana {a:>5}d   camelyon {b:>5}d   '
          f'{"ok" if a==b else "not matching"}')

silindi, yeniden cikarilacak


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

MIDNIGHT smoke test ok -> 1536d
  hospital 0: 14757x1536  (316s)
  hospital 1: 14956x1536  (409s)
  hospital 2: 15579x1536  (502s)
  hospital 3: 6450x1536  (595s)
  hospital 4: 9519x1536  (689s)

bitti, 11.5 dk

=== iki dataset arasi boyut kontrolu ===
  clip           ana   768d   camelyon   768d   ok
  plip           ana   768d   camelyon   768d   ok
  conch          ana   512d   camelyon   512d   ok
  uni            ana  1024d   camelyon  1024d   ok
  virchow2       ana  2560d   camelyon  2560d   ok
  gigapath       ana  1536d   camelyon  1536d   ok
  h_optimus_0    ana  1536d   camelyon  1536d   ok
  midnight       ana  1536d   camelyon  1536d   ok


## What part 2 does

With these embeddings cached, the transfer protocol is CPU-only:

- **in-distribution**, 5 hospitals x 8 models x 6 label fractions x 3 seeds
- **transfer**, all 20 directed hospital pairs on the same grid, temperature
  fitted on the source hospital's validation split
- the reverse-scaling test applied to 160 (model, pair) combinations rather
  than 32
- OOD_Stability computed on pairs that share a label definition, which is the
  quantity Equation 1 was meant to express

If the effect appears here, it belongs to covariate shift rather than to the
label-definition change, and Section 4 can drop the qualification the current
design forces. If it does not, that bounds the claim to label-definition shift,
which is also worth knowing and is what the submitted pairs actually test.